# Exploration of Bluesky API pipeline

In [1]:
import asyncio
import aiohttp
import re
from datetime import datetime, timezone, timedelta

from atproto import AsyncFirehoseSubscribeReposClient, parse_subscribe_repos_message, models, IdResolver, CAR
from atproto.exceptions import FirehoseError

In [2]:
ARAMCO_KEYWORDS = ["aramco", "samref", "yanbu", "ras tanura", "oil facility"]
regex_pattern = re.compile(rf"\b({'|'.join(re.escape(w) for w in ARAMCO_KEYWORDS)})\b", re.IGNORECASE)
TRUSTED_DIDS = {"did:plc:vovinwhtulbsx4mwfw26r5ni", "did:plc:jz3umb574v5ixivurtelqstt", "did:plc:tshrll7hb5scyeg4m6nitxtr"}
MIN_ACCOUNT_AGE_DAYS = 0

captured_posts = []

resolver = IdResolver()

In [3]:
async def get_profile_age(session, did):
    """Fetches public profile data."""
    url = f"https://public.api.bsky.app/xrpc/app.bsky.actor.getProfile?actor={did}"
    try:
        async with session.get(url) as response:
            if response.status == 200:
                return await response.json()
    except Exception:
        pass
    return {}

In [4]:
async def on_message_handler(message, session):
    try:
        commit = parse_subscribe_repos_message(message)
        if not isinstance(commit, models.ComAtprotoSyncSubscribeRepos.Commit):
            return # Exit early if it's not a commit

        author_did = commit.repo

        try:
            car = CAR.from_bytes(commit.blocks)
        except Exception:
            return # Exit early if the cryptographic CAR file is malformed

        # Loop through the operations in the commit
        for op in commit.ops:
            
            # --- CHECK 1: Is it a new post? ---
            if op.action != "create" or not op.path.startswith("app.bsky.feed.post"):
                continue

            record = car.blocks.get(op.cid) 
            if not record:
                continue

            text = record.get('text', '').lower()

            # --- CHECK 2: Does it match our keywords? ---
            if not regex_pattern.search(text):
                continue

            # --- CHECK 3: Reputation / Account Age ---
            # Now that we know it has the keywords, we check the account age
            profile = await get_profile_age(session, author_did)
            created_at_str = profile.get("createdAt")
            
            if not created_at_str:
                continue
                
            created_at = datetime.fromisoformat(created_at_str.replace("Z", "+00:00"))
            age = datetime.now(timezone.utc) - created_at
            
            if age < timedelta(days=MIN_ACCOUNT_AGE_DAYS):
                continue

            if author_did not in TRUSTED_DIDS:
                continue

            # --- ALL CHECKS PASSED ---
            # If the code reaches this point, the post has survived the gauntlet.

            captured_posts.append(record)
            
            print(f"Author: {author_did} (Account Age: {age.days} days)")
            print(f"Post: {text}")
            print(f"Timestamp: {datetime.now().strftime('%H:%M:%S')}")
            print("-" * 40)

    except Exception:
        pass

In [5]:
async def monitor_aramco_secure():
    print("Connecting to Raw AT Protocol Firehose... Waiting for matches...")
    
    async with aiohttp.ClientSession() as session:
        client = AsyncFirehoseSubscribeReposClient()
        
        async def handler_wrapper(message):
            asyncio.create_task(on_message_handler(message, session))
            
        await client.start(handler_wrapper)

In [6]:
await monitor_aramco_secure()

Connecting to Raw AT Protocol Firehose... Waiting for matches...
Author: did:plc:jz3umb574v5ixivurtelqstt (Account Age: 0 days)
Post: saudi aramco oil refinery targeted by iranian drones
Timestamp: 23:32:12
----------------------------------------


CancelledError: 

In [8]:
display(captured_posts)

[{'text': 'Saudi Aramco oil refinery targeted by Iranian drones',
  '$type': 'app.bsky.feed.post',
  'embed': {'$type': 'app.bsky.embed.external',
   'external': {'uri': 'https://news.sky.com/story/bluesky-13514180',
    'thumb': {'ref': b'\x01U\x12 \xec2\xbb\x88\xdbIu\t\x10\xb3Dz\x87\x00\x86\x94\xd7>\x98s\xc3\xf5W5\x03\x9f\xfa\xb8\xf1T\x11t',
     'size': 588940,
     '$type': 'blob',
     'mimeType': 'image/jpeg'},
    'title': 'Saudi Aramco oil refinery targeted by Iranian drones',
    'description': 'The Ras Tanura oil refinery near Dammam was temporarily shut down while thick smoke could be seen rising from the site. Saudi state television reported there were no casualties from the fire, as tensi...'}},
  'langs': ['en'],
  'createdAt': '2026-03-20T15:32:06.991Z'}]

In [20]:
print(captured_posts[0]['text'])
print(captured_posts[0]['embed']['external']['uri'])
print(captured_posts[0]['embed']['external']['title'])
print(captured_posts[0]['embed']['external']['description'])
print(captured_posts[0]['createdAt'])

Saudi Aramco oil refinery targeted by Iranian drones
https://news.sky.com/story/bluesky-13514180
Saudi Aramco oil refinery targeted by Iranian drones
The Ras Tanura oil refinery near Dammam was temporarily shut down while thick smoke could be seen rising from the site. Saudi state television reported there were no casualties from the fire, as tensi...
2026-03-20T15:32:06.991Z
